In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


# 경로

BASE = Path("/")

SOURCE_DIR = BASE / ""
EMBEDDING_DIR = BASE / ""
OUTPUT_DIR = BASE / ""

MODALITIES = {
    "email": "email_labeled.csv",
    "http": "http_labeled.csv",
    "file": "file_labeled.csv",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# modality별 사용자·날짜 mean pooling

for modality, source_filename in MODALITIES.items():
    source_path = SOURCE_DIR / source_filename
    embedding_input_dir = EMBEDDING_DIR / modality
    modality_output_dir = OUTPUT_DIR / modality

    modality_output_dir.mkdir(parents=True, exist_ok=True)

    if not source_path.exists():
        raise FileNotFoundError(f"[{modality}] 원본 CSV가 없습니다: {source_path}")

    if not embedding_input_dir.exists():
        raise FileNotFoundError(f"[{modality}] embedding 폴더가 없습니다: {embedding_input_dir}")

    # PKL 생성 당시와 동일하게 user/date 기준 과거 → 현재 정렬
    events = pd.read_csv(source_path, usecols=["user", "date"], low_memory=False)
    events["date"] = pd.to_datetime(events["date"], errors="coerce")

    if events["user"].isna().any() or events["date"].isna().any():
        raise ValueError(f"[{modality}] user 또는 date 결측/파싱 실패가 있습니다.")

    events = events.sort_values(["user", "date"], kind="stable").reset_index(drop=True)

    total_events = 0
    total_days = 0

    for user, user_events in events.groupby("user", sort=False):
        embedding_path = embedding_input_dir / f"{modality}_{user}_embedding.pkl"

        if not embedding_path.exists():
            raise FileNotFoundError(f"[{modality}/{user}] embedding PKL이 없습니다: {embedding_path}")

        with open(embedding_path, "rb") as f:
            vectors = pickle.load(f)

        if not isinstance(vectors, np.ndarray) or vectors.ndim != 2:
            raise ValueError(f"[{modality}/{user}] embedding은 2차원 NumPy 배열이어야 합니다.")

        if vectors.shape[0] != len(user_events):
            raise ValueError(
                f"[{modality}/{user}] 행 수 불일치: "
                f"events={len(user_events):,}, embeddings={vectors.shape[0]:,}"
            )

        if not np.isfinite(vectors).all():
            raise ValueError(f"[{modality}/{user}] embedding에 NaN 또는 Inf가 있습니다.")


        day_codes, unique_days = pd.factorize(
            user_events["date"].dt.normalize(),
            sort=False
        )

        sums = np.zeros((len(unique_days), vectors.shape[1]), dtype=np.float32)
        np.add.at(sums, day_codes, vectors)

        counts = np.bincount(day_codes).astype(np.float32)
        mean_vectors = sums / counts[:, None]

        output = {
            "user": user,
            "modality": modality,
            "dates": pd.to_datetime(unique_days).strftime("%Y-%m-%d").tolist(),
            "embeddings": mean_vectors.astype(np.float32),
        }

        output_path = modality_output_dir / f"{modality}_{user}_meanpooled.pkl"

        with open(output_path, "wb") as f:
            pickle.dump(output, f, protocol=pickle.HIGHEST_PROTOCOL)

        # 저장 검증
        with open(output_path, "rb") as f:
            saved = pickle.load(f)

        if (
            saved["user"] != user
            or saved["modality"] != modality
            or len(saved["dates"]) != len(unique_days)
            or saved["embeddings"].shape != mean_vectors.shape
        ):
            raise RuntimeError(f"[{modality}/{user}] mean pooling PKL 저장 검증 실패")

        total_events += len(user_events)
        total_days += len(unique_days)

    print(
        f"[{modality.upper()}] events={total_events:,}, "
        f"user-days={total_days:,}, users={events['user'].nunique():,}, "
        f"saved_to={modality_output_dir}"
    )

print(f"\n모든 mean pooling 완료: {OUTPUT_DIR}")